In [14]:
#SETTING
from google.colab import userdata, drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Info
GIT_TOKEN = userdata.get("My_Token")
GIT_USERNAME = "Tanior14"
GIT_EMAIL = "thuantan1905@gmail.com"
REPO_OWNER = "nhnminh1409"
GIT_REPO = "instacart-market-basket-analysis"

# Define Source File and Destination Path
SOURCE_PATH = "/content/drive/MyDrive/Colab Notebooks/ 04_feature_engineering.ipynb"
TARGET_DIR = "notebooks"

# 3.Setup Git Repo
%cd /content/
# Fix URL formatting (Remove redundant ://)
PUSH_URL = f"https://{GIT_TOKEN}@github.com/{REPO_OWNER}/{GIT_REPO}.git"

if not os.path.exists(GIT_REPO):
    !git clone {PUSH_URL}
else:
    print("Repo existed!")

%cd {GIT_REPO}
!git config --global user.email "{GIT_EMAIL}"
!git config --global user.name "{GIT_USERNAME}"

!git fetch origin
!git reset --hard origin/main

# 4. Copy files to the specific directory
# Create 'notebooks' directory if not exists
!mkdir -p {TARGET_DIR}

# Move files to 'notebooks' directory
!cp "{SOURCE_PATH}" "{TARGET_DIR}/"

# 5. Deploy code to GitHub repository
!git add .
!git commit -m "feat: complete user behavior EDA and feature engineering" || echo "No changes to commit"
!git push origin main --force

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
Cloning into 'instacart-market-basket-analysis'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 95 (delta 50), reused 44 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 1.39 MiB | 10.72 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/instacart-market-basket-analysis
HEAD is now at 557fdc1 feat: complete user behavior EDA and feature engineering
[main c5e669f] feat: complete user behavior EDA and feature engineering
 1 file changed, 1 insertion(+), 1 deletion(-)
 rewrite notebooks/ 04_feature_engineering.ipynb (94%)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 18.02 KiB | 6.01 MiB/s, done.
To

# Instacart Market Basket Analysis
## Feature Engineering

**Dataset:** `prior_data_cleaned.parquet` — 32M rows of historical purchase data

This notebook constructs the complete feature matrix, divided into three parts:

| Part | Table Name | Description |
|------|----------|-------|
| 1 | `user_features` | Aggregated behavior for each individual customer |
| 2 | `product_features` | Statistical characteristics of each product |
| 3 | `user_product_features` | Interaction history between users and products |

> **Output:** Three .parquet files ready to be merged into the feature matrix for the predictive model.

In [4]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [5]:
DATA_PATH = '/content/drive/MyDrive/Instacart_Project/'

df = pd.read_parquet(DATA_PATH + 'prior_data_cleaned.parquet')

print(f'Loaded successfully!')
print(f'   Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

Loaded successfully!
   Shape : 32,432,247 rows × 15 columns
   Memory: 1042.9 MB


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,product_name,aisle_id,aisle,department_id,department,add_to_cart_order,reordered
0,2,202279,prior,3,5,9,8.0,33120,Organic Egg Whites,86,eggs,16,dairy eggs,1,1
1,2,202279,prior,3,5,9,8.0,28985,Michigan Organic Kale,83,fresh vegetables,4,produce,2,1
2,2,202279,prior,3,5,9,8.0,9327,Garlic Powder,104,spices seasonings,13,pantry,3,0
3,2,202279,prior,3,5,9,8.0,45918,Coconut Butter,19,oils vinegars,13,pantry,4,1
4,2,202279,prior,3,5,9,8.0,30035,Natural Sweetener,17,baking ingredients,13,pantry,5,0


In [6]:
def reduce_mem_usage(df, verbose=True):
    """Downcast numeric columns to reduce memory footprint."""
    start_mem = df.memory_usage(deep=True).sum() / 1e6
    for col in df.select_dtypes(include=['int64', 'float64']).columns:
        col_min, col_max = df[col].min(), df[col].max()
        if str(df[col].dtype).startswith('int'):
            if col_min >= np.iinfo(np.int8).min and col_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        else:
            df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage(deep=True).sum() / 1e6
    if verbose:
        print(f'Memory reduced: {start_mem:.1f} MB → {end_mem:.1f} MB ({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df

# Pre-compute order-level deduplication (used for all 3 parts)
orders_dedup = df.drop_duplicates(subset='order_id').copy()

# Basket size per order
basket = df.groupby('order_id')['product_id'].count().reset_index()
basket.columns = ['order_id', 'basket_size']
orders_dedup = orders_dedup.merge(basket, on='order_id', how='left')

print('Helper functions & shared tables ready.')
print(f'Orders table shape: {orders_dedup.shape}')

Helper functions & shared tables ready.
Orders table shape: (3214854, 16)


---
## PART 1 — USER FEATURES

each row = one `user_id`, aggregate full purchase history.

| Nhóm | Features |
|------|----------|
| Order stats | total_orders, avg/max/min basket size |
| Shopping cadence | avg/median/std days between orders |
| Temporal habits | peak hour, peak dow, avg hour, avg dow |
| Reorder behavior | reorder_rate, total_items, unique_products |
| Diversity | diversity_ratio |

In [7]:
print('Building USER FEATURES...')

# 1. Core order stats
uf = orders_dedup.groupby('user_id').agg(
    user_total_orders        = ('order_id',             'count'),
    user_avg_basket_size     = ('basket_size',           'mean'),
    user_max_basket_size     = ('basket_size',           'max'),
    user_min_basket_size     = ('basket_size',           'min'),
    user_avg_days_between    = ('days_since_prior_order', lambda x: x[x > 0].mean()),
    user_median_days_between = ('days_since_prior_order', lambda x: x[x > 0].median()),
    user_std_days_between    = ('days_since_prior_order', lambda x: x[x > 0].std()),
    # Temporal habits
    user_peak_hour           = ('order_hour_of_day',    lambda x: x.value_counts().idxmax()),
    user_peak_dow            = ('order_dow',            lambda x: x.value_counts().idxmax()),
    user_avg_hour            = ('order_hour_of_day',    'mean'),
    user_avg_dow             = ('order_dow',            'mean'),
).reset_index()

# 2. Reorder & product stats
reorder_stats = df.groupby('user_id').agg(
    user_reorder_rate       = ('reordered',   'mean'),
    user_total_items_bought = ('product_id',  'count'),
    user_unique_products    = ('product_id',  'nunique'),
).reset_index()

uf = uf.merge(reorder_stats, on='user_id', how='left')

# 3. Derived features
uf['user_diversity_ratio'] = (
    uf['user_unique_products'] / uf['user_total_items_bought']
).astype(np.float32)

# 4. Memory optimization
uf = reduce_mem_usage(uf)

print(f'\n user_features shape: {uf.shape}')
uf.head()

Building USER FEATURES...
Memory reduced: 21.9 MB → 9.3 MB (57.5% reduction)

 user_features shape: (206209, 16)


,user_id,user_total_orders,user_avg_basket_size,user_max_basket_size,user_min_basket_size,user_avg_days_between,user_median_days_between,user_std_days_between,user_peak_hour,user_peak_dow,user_avg_hour,user_avg_dow,user_reorder_rate,user_total_items_bought,user_unique_products,user_diversity_ratio
0,1,10,5.900000,9,4,22.000000,20.5,6.279217,7,4,10.300000,2.500000,0.694915,59,18,0.305085
1,2,14,13.928572,26,5,15.230769,13.0,9.867064,10,2,10.571428,2.142857,0.476923,195,102,0.523077
2,3,12,7.333333,11,5,12.090909,11.0,5.375026,16,0,16.416666,1.083333,0.625000,88,33,0.375000
3,4,5,3.600000,7,2,18.333334,19.0,3.055050,13,5,12.600000,4.800000,0.055556,18,17,0.944444
4,5,4,9.250000,12,5,13.333333,11.0,4.932883,18,3,16.000000,1.750000,0.378378,37,23,0.621622


In [11]:
# Quick stats
uf.describe().T.style.background_gradient(cmap='Greens', subset=['mean', 'std'])

,count,mean,std,min,25%,50%,75%,max
user_id,206209.000000,103105.000000,59527.555167,1.000000,51553.000000,103105.000000,154657.000000,206209.000000
user_total_orders,206209.000000,15.590270,16.654649,3.000000,5.000000,9.000000,19.000000,99.000000
user_avg_basket_size,206209.000000,9.951235,5.861808,1.000000,5.740741,8.933333,13.000000,70.250000
user_max_basket_size,206209.000000,17.652508,10.176176,1.000000,10.000000,16.000000,23.000000,100.000000
user_min_basket_size,206209.000000,4.294735,3.953049,1.000000,1.000000,3.000000,6.000000,51.000000
user_avg_days_between,206185.000000,15.404814,7.109001,1.000000,9.571428,14.722222,20.500000,30.000000
user_median_days_between,206185.000000,14.927140,8.666127,1.000000,7.000000,13.000000,21.500000,30.000000
user_std_days_between,205565.000000,7.279368,3.690078,0.000000,4.645787,7.671354,9.812529,20.506096
user_peak_hour,206209.000000,13.515075,3.918294,0.000000,10.000000,13.000000,16.000000,23.000000
user_peak_dow,206209.000000,2.553996,2.136871,0.000000,1.000000,2.000000,5.000000,6.000000


In [ ]:
# Export
OUT_USER = DATA_PATH + 'user_features.parquet'
uf.to_parquet(OUT_USER, index=False)

print(f'Saved → {OUT_USER}')
print(f'   Shape  : {uf.shape[0]:,} users × {uf.shape[1]} features')
print(f'   Memory : {uf.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('Feature list:')
for col in uf.columns:
    print(f'   {col:<35} ({uf[col].dtype})')

---
## Part 2 · Product Features

We compute product-level signals covering two complementary angles:

- **Reorder rate** – how loyal customers are to a product
- **First-in-Basket (FiB) rate** – how often a product is the very first item added to the cart

In [8]:
print('Building PRODUCT FEATURES...')

# 1. Core product stats
pf = df.groupby('product_id').agg(
    prod_total_orders       = ('order_id',           'count'),
    prod_total_users        = ('user_id',            'nunique'),
    prod_reorder_rate       = ('reordered',          'mean'),
    prod_avg_add_to_cart    = ('add_to_cart_order',  'mean'),
    prod_median_add_to_cart = ('add_to_cart_order',  'median'),
    prod_avg_order_hour     = ('order_hour_of_day',  'mean'),
    prod_avg_order_dow      = ('order_dow',          'mean'),
).reset_index()

# 2. Reorder rate features
# Total reorders and total purchases per product → explicit reorder count
reorder_agg = df.groupby('product_id').agg(
    prod_total_reorders = ('reordered', 'sum'),   # absolute number of reorders
).reset_index()

pf = pf.merge(reorder_agg, on='product_id', how='left')
pf['prod_total_reorders'] = pf['prod_total_reorders'].fillna(0)

#  3. First-in-Basket (FiB) features
# Count total purchases per product (denominator)
total_purchases = (
    df.groupby('product_id')
    .size()
    .reset_index(name='prod_total_purchases')
)

# Count how many times the product was added FIRST to the cart
first_in_basket = (
    df[df['add_to_cart_order'] == 1]
    .groupby('product_id')
    .size()
    .reset_index(name='prod_first_in_basket_count')
)

fib_df = total_purchases.merge(first_in_basket, on='product_id', how='left')
fib_df['prod_first_in_basket_count'] = fib_df['prod_first_in_basket_count'].fillna(0)

# First-in-Basket rate = first-cart orders / total purchases
fib_df['prod_first_in_basket_rate'] = (
    fib_df['prod_first_in_basket_count'] / fib_df['prod_total_purchases']
).astype(np.float32)

pf = pf.merge(
    fib_df[['product_id', 'prod_first_in_basket_count', 'prod_first_in_basket_rate']],
    on='product_id',
    how='left'
)

# 4. Category info (department & aisle)
cat_info = df[['product_id', 'department', 'aisle']].drop_duplicates('product_id')
pf = pf.merge(cat_info, on='product_id', how='left')

# 5. Derived: purchase concentration (orders / unique users)
pf['prod_orders_per_user'] = (
    pf['prod_total_orders'] / pf['prod_total_users']
).astype(np.float32)

# 6. Memory optimization
pf = reduce_mem_usage(pf)

print(f'\n product_features shape: {pf.shape}')
pf.head()

Building PRODUCT FEATURES...
Memory reduced: 4.3 MB → 2.5 MB (41.2% reduction)

✅ product_features shape: (49677, 14)


,product_id,prod_total_orders,prod_total_users,prod_reorder_rate,prod_avg_add_to_cart,prod_median_add_to_cart,prod_avg_order_hour,prod_avg_order_dow,prod_total_reorders,prod_first_in_basket_count,prod_first_in_basket_rate,department,aisle,prod_orders_per_user
0,1,1852,716,0.613391,5.801836,4.0,13.238121,2.776458,1136,272.0,0.146868,snacks,cookies cakes,2.586592
1,2,90,78,0.133333,9.888889,8.0,13.277778,2.922222,12,11.0,0.122222,pantry,spices seasonings,1.153846
2,3,277,74,0.732852,6.415163,4.0,12.104693,2.736462,203,43.0,0.155235,beverages,tea,3.743243
3,4,329,182,0.446809,9.507599,8.0,13.714286,2.683891,147,14.0,0.042553,frozen,frozen meals,1.807692
4,5,15,6,0.600000,6.466667,6.0,10.666667,2.733333,9,0.0,0.000000,pantry,marinades meat preparation,2.500000


In [9]:
# Quick stats
pf.describe().T.style.background_gradient(cmap='Greens', subset=['mean', 'std'])

,count,mean,std,min,25%,50%,75%,max
product_id,49677.000000,24843.417356,14343.034804,1.000000,12423.000000,24842.000000,37264.000000,49688.000000
prod_total_orders,49677.000000,652.862431,4791.975330,1.000000,17.000000,60.000000,260.000000,472561.000000
prod_total_users,49677.000000,267.878636,1308.767995,1.000000,11.000000,35.000000,137.000000,73956.000000
prod_reorder_rate,49677.000000,0.366459,0.208104,0.000000,0.207792,0.376623,0.529290,0.941176
prod_avg_add_to_cart,49677.000000,9.093280,2.546200,1.000000,7.625000,9.053254,10.351159,53.000000
prod_median_add_to_cart,49677.000000,7.256658,2.658107,1.000000,6.000000,7.000000,8.500000,53.000000
prod_avg_order_hour,49677.000000,13.504277,1.058930,0.000000,13.066667,13.500000,13.944445,23.000000
prod_avg_order_dow,49677.000000,2.858459,0.506675,0.000000,2.636514,2.822222,3.058824,6.000000
prod_total_reorders,49677.000000,384.989693,3601.611844,0.000000,4.000000,22.000000,115.000000,398605.000000
prod_first_in_basket_count,49677.000000,64.715141,749.441284,0.000000,1.000000,5.000000,22.000000,110916.000000


---
## Part 3 · User–Product Interaction Features

We join user-level and product-level signals with per-pair purchase history to form an interaction table. Features capture:

- **Frequency** – how many times a user purchased a specific product
- **Recency** – how recently the product was bought (last order number)
- **Purchase ratio** – share of a user's total basket occupied by that product
- **Reorder probability** – reorder rate of this pair
- **Relative position** – how early the product is added to the cart vs. user habit
- **Enriched context** – joined user & product statistics as contextual features

In [12]:
print('Building USER–PRODUCT INTERACTION FEATURES...')

# 1. Core pair-level aggregations
upf = df.groupby(['user_id', 'product_id']).agg(
    up_times_bought        = ('order_number',      'count'),   # frequency
    up_last_order_number   = ('order_number',      'max'),     # recency
    up_first_order_number  = ('order_number',      'min'),     # first encounter
    up_reorder_rate        = ('reordered',         'mean'),    # reorder probability
    up_avg_add_to_cart     = ('add_to_cart_order', 'mean'),    # avg cart position
    up_min_add_to_cart     = ('add_to_cart_order', 'min'),     # best (earliest) position
).reset_index()

# 2. User-level totals (needed for ratio features)
user_total_items  = (
    df.groupby('user_id')
    .size()
    .reset_index(name='user_total_items')
)
user_total_orders = (
    df.groupby('user_id')['order_number']
    .max()
    .reset_index(name='user_total_orders')
)

upf = upf.merge(user_total_items,  on='user_id', how='left')
upf = upf.merge(user_total_orders, on='user_id', how='left')

# 3. Pair-level derived metrics
# Preference: share of user's total basket occupied by this product
upf['up_purchase_ratio'] = (
    upf['up_times_bought'] / upf['user_total_items']
).astype(np.float32)

# Recency score: how recently the pair was active (higher = more recent)
upf['up_recency_score'] = (
    upf['up_last_order_number'] / upf['user_total_orders']
).astype(np.float32)

# Order span: how long the product has been in the user's repertoire
upf['up_order_span'] = (
    upf['up_last_order_number'] - upf['up_first_order_number']
).astype(np.int16)

# 4. Enrich with user features
user_cols = [
    'user_id',
    'user_total_orders',
    'user_avg_basket_size',
    'user_reorder_rate',
    'user_unique_products',
    'user_diversity_ratio',
    'user_avg_days_between',
    'user_peak_hour',
    'user_peak_dow',
]
upf = upf.merge(uf[user_cols], on='user_id', how='left')

# 5. Enrich with product features
prod_cols = [
    'product_id',
    'prod_total_orders',
    'prod_reorder_rate',
    'prod_avg_add_to_cart',
    'prod_first_in_basket_rate',
    'prod_orders_per_user',
    'department',
    'aisle',
]
upf = upf.merge(pf[prod_cols], on='product_id', how='left')

# 6. Cross features: user habit vs. product behaviour
# How much earlier/later this user adds the product vs. the product's global average
upf['up_cart_position_vs_prod_avg'] = (
    upf['up_avg_add_to_cart'] - upf['prod_avg_add_to_cart']
).astype(np.float32)

# User reorder rate relative to the product's global reorder rate
upf['up_reorder_rate_vs_prod'] = (
    upf['up_reorder_rate'] - upf['prod_reorder_rate']
).astype(np.float32)

# Drop helper columns no longer needed
upf = upf.drop(columns=['user_total_items', 'user_total_orders'], errors='ignore')

#7. Memory optimization
upf = reduce_mem_usage(upf)

print(f'\n user_product_features shape: {upf.shape}')
upf.head()

Building USER–PRODUCT INTERACTION FEATURES...
Memory reduced: 1304.1 MB → 1104.5 MB (15.3% reduction)

 user_product_features shape: (13307407, 29)


,user_id,product_id,up_times_bought,up_last_order_number,up_first_order_number,up_reorder_rate,up_avg_add_to_cart,up_min_add_to_cart,user_total_orders_x,up_purchase_ratio,...,user_peak_dow,prod_total_orders,prod_reorder_rate,prod_avg_add_to_cart,prod_first_in_basket_rate,prod_orders_per_user,department,aisle,up_cart_position_vs_prod_avg,up_reorder_rate_vs_prod
0,1,196,10,10,1,0.900000,1.400000,1,10,0.169492,...,4,35791,0.776480,3.721774,0.328854,4.473875,beverages,soft drinks,-2.321774,0.123520
1,1,10258,9,10,2,0.888889,3.333333,2,10,0.152542,...,4,1946,0.713772,4.277493,0.191675,3.493716,snacks,nuts seeds dried fruit,-0.944159,0.175117
2,1,10326,1,5,5,0.000000,5.000000,5,10,0.016949,...,4,5526,0.652009,4.191097,0.189468,2.873635,produce,fresh fruits,0.808903,-0.652009
3,1,12427,10,10,1,0.900000,3.300000,1,10,0.169492,...,4,6476,0.740735,4.760037,0.199043,3.857058,snacks,popcorn jerky,-1.460037,0.159265
4,1,13032,3,10,2,0.666667,6.333333,5,10,0.050847,...,4,3751,0.657158,5.622767,0.155692,2.916796,breakfast,cereal,0.710566,0.009509


In [13]:
# Quick stats
upf.describe().T.style.background_gradient(cmap='Oranges', subset=['mean', 'std'])

,count,mean,std,min,25%,50%,75%,max
user_id,13307407.000000,103000.822643,59436.699106,1.000000,51584.000000,102717.000000,154453.000000,206209.000000
product_id,13307407.000000,25513.498638,14224.299514,1.000000,13292.000000,25640.000000,38157.000000,49688.000000
up_times_bought,13307407.000000,2.437158,3.554271,1.000000,1.000000,1.000000,2.000000,99.000000
up_last_order_number,13307407.000000,15.867484,17.302995,1.000000,4.000000,9.000000,21.000000,99.000000
up_first_order_number,13307407.000000,10.691836,13.479012,1.000000,2.000000,6.000000,13.000000,99.000000
up_reorder_rate,13307407.000000,0.265511,0.348477,0.000000,0.000000,0.000000,0.500000,1.000000
up_avg_add_to_cart,13307407.000000,9.214161,6.895164,1.000000,4.000000,7.500000,12.000000,100.000000
up_min_add_to_cart,13307407.000000,7.690545,7.063995,1.000000,2.000000,6.000000,11.000000,100.000000
user_total_orders_x,13307407.000000,25.380333,22.280979,3.000000,9.000000,18.000000,35.000000,99.000000
up_purchase_ratio,13307407.000000,0.015496,0.023551,0.000268,0.003831,0.008403,0.018182,1.000000


---
## Summary

| Feature table | Shape | Key signals |
|---|---|---|
| `uf` – user features | `(n_users, 14)` | basket size, order frequency, reorder rate, product diversity, temporal habits |
| `pf` – product features | `(n_products, 14)` | reorder rate, total orders, first-in-basket rate, cart position, dept/aisle |
| `upf` – user–product interaction | `(n_pairs, 20+)` | purchase frequency, recency, preference ratio, enriched user & product context |

In [ ]:
# ── Optional: save all feature tables ────────────────────────────────────────
# uf.to_parquet('user_features.parquet',         index=False)
# pf.to_parquet('product_features.parquet',      index=False)
# upf.to_parquet('user_product_features.parquet', index=False)

# print(' All feature tables saved.')